**Process Addresses Data**

1. Ingest the data into the data lakehouse -  bronze_addresses
2. Perform data quality checks and transform the data as required. -  silver_addresses_clean
3. Apply changes to the addresses Data (SCD Type 2) - silver_addresses

**1.Ingest the data into the data lakehouse - bronze_addresses**

In [0]:
import dlt
import pyspark.sql.functions as F


In [0]:
@dlt.table(
    name = "bronze_addresses",
    table_properties = {'qality':'bronze'},
    comment = "This is the bronze table for addresses"
)
def create_bronze_addresses():
    return(
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.inferColumnTypes", "true")
        .load("/Volumes/circuitbox/landing/operational_data/addresses/")
        .select(
            "*",
            F.col("_metadata.file_path").alias("file_path"),
            F.current_timestamp().alias("ingest_timestamp")
        )


        )
       
    


**2.Perform data quality checks and transform the data as required. - silver_addresses_clean**

In [0]:
@dlt.table(
        name = "silver_addresses_clean",
        table_properties = {'qality':'silver'},
        comment = "This is the silver table for addresses"
)
@dlt.expect_or_fail("valid_customer_id","customer_id is not null")
@dlt.expect_or_drop("valid_address_line_1","address_line_1 is not null")
@dlt.expect("valid_postcode","length(postcode) == 5")

def create_silver_addresses_clean():
    return(
        spark.readStream.table("bronze_addresses")
            .select(
                "customer_id",
                "address_line_1",
                "city",
                "state",
                "postcode",
                F.col("created_date").cast("date").alias("created_date")

            )          
    )
     

**3. Apply changes to the addresses Data (SCD Type 2) - silver_addresses**

In [0]:
dlt.create_streaming_table(
    name = "silver_addresses",
    comment="This is the silver table for addresses",
    table_properties= {"quality" : "silver"}
)

In [0]:
dlt.apply_changes(
    target = "silver_addresses",
    source = "silver_addresses_clean",
    keys = ["customer_id"],
    sequence_by = "created_date",
    stored_as_scd_type = 2
)